# DBSCAN — Euclidean


In [ ]:
import time
import tracemalloc
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.cluster import DBSCAN
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler, Normalizer

# Path portabel: saat membuka notebook langsung, gunakan folder berisi cafes.csv
# sebagai working directory kernel, atau ubah path default di bawah ini.
# Runner summarize_notebooks.py memasok kedua path ini secara otomatis.
DATA_PATH = Path(globals().get("DATA_PATH", "cafes.csv")).expanduser().resolve()
FIGURES_DIR = Path(globals().get(
    "FIGURES_DIR", DATA_PATH.parent / "reports" / "figures"
)).expanduser().resolve()
if not DATA_PATH.is_file():
    raise FileNotFoundError(
        f"CSV tidak ditemukan: {DATA_PATH}. Letakkan cafes.csv di folder kerja "
        "kernel atau ubah DATA_PATH pada sel ini."
    )
FIGURES_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
# ==============================================================================
# BAGIAN A: LOAD DATA & PREPROCESSING
# ==============================================================================
print("\n" + "="*56)
print("  BAGIAN A: LOAD DATA & PREPROCESSING (DBSCAN)")
print("="*56)

df = pd.read_csv(DATA_PATH, on_bad_lines="skip")
kolom_fitur = ['latitude', 'longitude', 'rating', 'review_count']
kolom_ada = [k for k in kolom_fitur if k in df.columns]
df_fitur = df[kolom_ada].dropna()

# Standardisasi fitur numerik
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df_fitur)

print(f"Data siap | Shape: {X_scaled.shape} | Baris: {len(df_fitur)}")


In [ ]:
# ==============================================================================
# BAGIAN B: K-DISTANCE GRAPH (UNTUK MENENTUKAN EPSILON OPTIMAL)
# ==============================================================================
print("\n" + "="*56)
print("  BAGIAN B: K-DISTANCE GRAPH (KNN UNTUK TUNING EPS)")
print("="*56)

# Nilai min_samples acuan untuk estimasi eps
K_NEIGHBORS = 4

# 1. K-Distance Euclidean
nn_e = NearestNeighbors(n_neighbors=K_NEIGHBORS, metric='euclidean').fit(X_scaled)
d_e, _ = nn_e.kneighbors(X_scaled)
kdist_e = np.sort(d_e[:, -1])[::-1]
EPS_EUCLID = 1.0

plt.figure(figsize=(6, 4.5))
plt.plot(kdist_e, color='navy', linewidth=2)
plt.axhline(y=EPS_EUCLID, color='red', linestyle='--', linewidth=1.5, label=f'eps={EPS_EUCLID}')
plt.title(f'K-Distance Graph (Euclidean, k={K_NEIGHBORS})')
plt.xlabel('Titik Data (diurutkan)')
plt.ylabel(f'Jarak Euclidean ke tetangga ke-{K_NEIGHBORS}')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.5)

plt.tight_layout()
kdist_plot_path = FIGURES_DIR / "dbscan_euclidean_kdistance_plots.png"
plt.savefig(kdist_plot_path, dpi=150)
plt.show()
plt.close()
print(f"K-distance graph disimpan: {kdist_plot_path}")


In [ ]:
# --- 1. DBSCAN EUCLIDEAN ---
print("\n" + "="*56)
print("  DBSCAN — METRIK: EUCLIDEAN")
print(f"  Parameter: eps={EPS_EUCLID}, min_samples={K_NEIGHBORS}")
print("="*56)

tracemalloc.start()
t_mulai = time.time()

db_euclid = DBSCAN(eps=EPS_EUCLID, min_samples=K_NEIGHBORS, metric='euclidean')
lbl_euclid = db_euclid.fit_predict(X_scaled)

t_selesai = time.time()
_, mem_puncak = tracemalloc.get_traced_memory()
tracemalloc.stop()

n_klaster_e = len(set(lbl_euclid)) - (1 if -1 in lbl_euclid else 0)
n_noise_e = int((lbl_euclid == -1).sum())
pct_noise_e = n_noise_e / len(lbl_euclid) * 100
mask_valid_e = lbl_euclid != -1

if mask_valid_e.sum() > 1 and n_klaster_e >= 2:
    sil_euclid = silhouette_score(X_scaled[mask_valid_e], lbl_euclid[mask_valid_e], metric='euclidean')
    sil_e_str = f"{sil_euclid:.4f}"
else:
    sil_euclid = None
    sil_e_str = "N/A (klaster < 2)"

waktu_e = t_selesai - t_mulai
mem_e_kb = mem_puncak / 1024
mem_e_mb = mem_puncak / (1024**2)

print(f"  Jumlah Klaster    : {n_klaster_e}")
print(f"  Titik Noise (-1)  : {n_noise_e} ({pct_noise_e:.1f}%)")
print(f"  Waktu Eksekusi    : {waktu_e:.4f} detik")
print(f"  Memori Puncak     : {mem_e_kb:.2f} KB ({mem_e_mb:.4f} MB)")
print(f"  Silhouette Score  : {sil_e_str}")
print("="*56)


In [ ]:
# ==============================================================================
# BAGIAN D: VISUALISASI PCA 2D DENGAN TITIK NOISE HITAM (X)
# ==============================================================================
print("\n" + "="*56)
print("  BAGIAN D: VISUALISASI PCA 2D (DBSCAN)")
print("="*56)

pca = PCA(n_components=2, random_state=42)
X_pca_std = pca.fit_transform(X_scaled)
v1_std, v2_std = pca.explained_variance_ratio_ * 100

fig, ax = plt.subplots(figsize=(6, 5.5), dpi=150)
axes = [ax]

# Helper untuk plot DBSCAN PCA (mengadaptasi cell visualisasi DBScan kuis_01)
def plot_dbscan_pca(ax, X_pca_coords, lbls, v1, v2, title):
    mask_noise = lbls == -1
    mask_klaster = ~mask_noise

    # Plot titik noise -> hitam, tanda x
    ax.scatter(X_pca_coords[mask_noise, 0], X_pca_coords[mask_noise, 1],
               c='black', marker='x', s=35, alpha=0.7,
               label=f'Noise ({mask_noise.sum()} titik)')

    # Plot titik klaster -> berwarna
    if mask_klaster.sum() > 0:
        sc = ax.scatter(X_pca_coords[mask_klaster, 0], X_pca_coords[mask_klaster, 1],
                        c=lbls[mask_klaster], cmap='Set1', s=30, alpha=0.8,
                        label='Klaster')
        plt.colorbar(sc, ax=ax, label='Klaster')

    ax.set_title(title, fontweight='bold')
    ax.set_xlabel(f'PCA 1 ({v1:.1f}%)')
    ax.set_ylabel(f'PCA 2 ({v2:.1f}%)')
    ax.legend(fontsize=9)
    ax.grid(True, linestyle='--', alpha=0.4)

plot_dbscan_pca(axes[0], X_pca_std, lbl_euclid, v1_std, v2_std,
                f"DBSCAN Euclidean\n{n_klaster_e} klaster, {n_noise_e} noise | Sil: {sil_e_str}")

plt.suptitle("Visualisasi PCA 2D DBSCAN (Data Numerik Kafe)", fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
dbscan_pca_path = FIGURES_DIR / "dbscan_euclidean_pca_plots.png"
plt.savefig(dbscan_pca_path, dpi=150, bbox_inches='tight')
plt.show()
plt.close()
print(f"Visualisasi PCA disimpan ke: {dbscan_pca_path}")


In [ ]:
# Dictionary ringkasan performa untuk pelaporan
DBSCAN_SUMMARY = {
    "euclidean": {
        "algo": "DBSCAN", "metric": "Euclidean", "param": f"eps={EPS_EUCLID}, ms={K_NEIGHBORS}",
        "time": waktu_e, "mem_mb": mem_e_mb, "clusters": n_klaster_e,
        "noise_pct": pct_noise_e, "silhouette": sil_euclid, "labels": lbl_euclid
    }
}
